# 5. ADULT (Census Income) - Preprocesamiento
**Tipo:** CLASIFICACIÓN BINARIA (predecir ingresos >50K)
**Variable objetivo:** income (>50K = 1, <=50K = 0)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CARGAR DATOS (UCI - sin header)
# ============================================================
# Nombres de columnas según documentación
columnas = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
            'marital-status', 'occupation', 'relationship', 'race', 'sex',
            'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

df = pd.read_csv('adult.data', names=columnas, skipinitialspace=True)
print(f"Dimensiones: {df.shape}")
df.head()

In [ ]:
# ============================================================
# ⚠️ LIMPIEZA DE NULOS (están como "?")
# ============================================================
# En este dataset los nulos aparecen como "?"
df = df.replace('?', np.nan)

print("Nulos por columna:")
print(df.isnull().sum())

# Opción 1: Eliminar filas con nulos (simple)
df = df.dropna()
print(f"\nFilas después de eliminar nulos: {len(df)}")

In [ ]:
# ============================================================
# PREPARAR VARIABLE OBJETIVO
# ============================================================
# Convertir >50K a 1, <=50K a 0
df['income'] = df['income'].map({'>50K': 1, '<=50K': 0, '>50K.': 1, '<=50K.': 0})
y = df['income'].values

print(f"Balance de clases: {np.bincount(y)}")
print(f"Proporción clase 1: {y.mean():.2%}")

In [ ]:
# ============================================================
# ELIMINAR COLUMNAS REDUNDANTES/NO ÚTILES
# ============================================================
# fnlwgt: peso poblacional, no predice ingresos individuales
# education: redundante con education-num

X_df = df.drop(columns=['income', 'fnlwgt', 'education'])
print(f"Columnas: {list(X_df.columns)}")

In [ ]:
# ============================================================
# ENCODING DE VARIABLES
# ============================================================
# Identificar tipos
cat_cols = X_df.select_dtypes(include=['object']).columns.tolist()
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Numéricas ({len(num_cols)}): {num_cols}")

# Opción simple: native-country tiene muchas categorías, simplificar
X_df['native-country'] = X_df['native-country'].apply(
    lambda x: 'USA' if x == 'United-States' else 'Other'
)

# One-Hot Encoding
X_encoded = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
print(f"\nDimensiones después de encoding: {X_encoded.shape}")

In [ ]:
# ============================================================
# DIVISIÓN DE DATOS
# ============================================================
X = X_encoded.to_numpy()

# Stratify para mantener balance de clases
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.333, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape[0]}")
print(f"Val: {X_val.shape[0]}")
print(f"Test: {X_test.shape[0]}")

In [ ]:
# ============================================================
# NORMALIZACIÓN
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("✅ Datos listos para CLASIFICACIÓN BINARIA")
print(f"Características: {X_train.shape[1]}")

## MÉTODO TRADICIONAL

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Regresión Logística
modelo_lr = LogisticRegression(max_iter=1000)
modelo_lr.fit(X_train, y_train)
y_pred = modelo_lr.predict(X_test)

print("=== Regresión Logística ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

## CON PYTORCH

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)

# Manejar desbalance de clases
pos_weight = torch.tensor([len(y_train) / y_train.sum() - 1])

class ClasificadorBinario(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.red(x)

modelo = ClasificadorBinario(X_train_t.shape[1])
criterio = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # Maneja desbalance
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)
print(modelo)